# Sistem Informasi Reservasi & Rental Armada Kendaraan
> **Domain**: Web & Sistem Informasi / Fleet Management | **Tech Stack**: React, Next.js, Modern JS, Tailwind Solid Dark
> **Author**: [Arjuna Fransesco](https://github.com/ArjunaFransesco) | **GitHub**: [Portfolio Repositories](https://github.com/ArjunaFransesco?tab=repositories)

---
## Executive Summary
🚗 Vehicle Fleet Booking & Rental Management System with React, Next.js, Modern JS, Interactive Fleet Calendar & Dynamic Pricing Engine

---
## Machine Learning Workflow
1. **Domain Understanding & Dataset Ingestion**
2. **Exploratory Data Analysis (EDA) & Feature Distribution**
3. **Preprocessing Pipeline & Feature Engineering**
4. **Model Architecture & Training**
5. **Evaluation, Error Metrics & Performance Visualization**
6. **Deployment Artifact Export**


### 1. Environment Setup & Core Library Imports
Importing data manipulation, statistical modeling, machine learning, and visualization libraries.


In [1]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# Plotting styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
print("Libraries loaded successfully!")


### 2. Dataset Ingestion & Exploratory Data Inspection
Loading the raw domain telemetry dataset and inspecting schema structures, null values, and summary statistics.


In [1]:
DATA_PATH = "../data/raw/sim_rental_kendaraan_fleet_dataset.csv"
df = pd.read_csv(DATA_PATH)
print(f"Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(10)


In [1]:
df.info()
df.describe().T


### 3. Exploratory Data Analysis (EDA) & Feature Distributions
Analyzing correlation patterns and target relationship across input features.


In [1]:
plt.figure(figsize=(9, 7))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True)
plt.title("Multivariate Correlation Heatmap", fontsize=14, fontweight="bold")
plt.show()


In [1]:
# Target Distribution
plt.figure(figsize=(8, 4))
sns.histplot(df["component_failure_risk"], kde=True, color="#3b82f6", bins=25)
plt.title(f"Distribution of Target: component_failure_risk", fontsize=13, fontweight="bold")
plt.show()


### 4. Data Preprocessing & Pipeline Transformation
Splitting dataset into stratified train and test partitions, followed by standard scaling normalization.


In [1]:
features = ["engine_rpm_mean", "engine_coolant_temp_c", "oil_pressure_psi", "hard_braking_events_per_100km", "battery_voltage_v", "odometer_reading_km_k"]
X = df[features]
y = df["component_failure_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Partition: {{X_train.shape}}")
print(f"Testing Partition: {{X_test.shape}}")


### 5. Machine Learning Model Training & 5-Fold Cross Validation
Instantiating ensemble architecture and performing cross-validation to assess generalization stability.


In [1]:
model = RandomForestClassifier(n_estimators=180, max_depth=10, random_state=42)
model.fit(X_train_scaled, y_train)

cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring="accuracy")
print(f"5-Fold CV Scores: {{cv_scores}}")
print(f"Mean CV Score: {{np.mean(cv_scores):.4f}} (±{{np.std(cv_scores):.4f}})")


### 6. Benchmark Evaluation & Error Metrics
Computing production-grade test evaluation metrics.


In [1]:
preds = model.predict(X_test_scaled)
acc = accuracy_score(y_test, preds)
print(f"Test Accuracy: {acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, preds))


### 7. Feature Importance Ranking
Quantifying the influence of each domain predictor.


In [1]:
feat_importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)
plt.figure(figsize=(9, 5))
feat_importances.plot(kind="barh", color="#0284c7")
plt.title("Relative Feature Importances (Gini/Impurity Reduction)", fontsize=13, fontweight="bold")
plt.xlabel("Importance Score")
plt.show()


### 8. Serialization & Real-Time Inference Demonstration
Saving the trained pipeline artifact and simulating single-sample inference.


In [1]:
sample_input = pd.DataFrame([X_test.iloc[0].to_dict()])
sample_scaled = scaler.transform(sample_input)
pred_val = model.predict(sample_scaled)[0]

print("Sample Telemetry Input:")
print(sample_input.to_dict(orient="records")[0])
print(f"\nModel Predicted Value (component_failure_risk): {pred_val}")
print(f"Ground Truth Label: {y_test.iloc[0]}")
